In [9]:
import pandas as pd

In [10]:
df = pd.read_excel("result_mcp.xlsx")

In [11]:
df

,Question,Model,Run,Error,Count,Note,Latency,Type
0,CML complete targets,gpt-4.1-nano,1,Upstream Retrieval Failure,0.0,NaN,17.8,Agentic
1,CML complete targets,gpt-4.1-nano,2,Upstream Retrieval Failure,0.0,GraphQL API Error,18.7,Agentic
2,CML complete targets,gpt-4o-mini,1,MaxTurnsExceeded,0.0,NaN,36.8,Agentic
3,CML complete targets,gpt-4o-mini,2,Upstream Retrieval Failure,0.0,Open Targets Access Error,35.8,Agentic
4,CML complete targets,gpt-4.1-mini,1,Upstream Service Availability Error,0.0,Open Targets API Unresponsive,28.8,Agentic
...,...,...,...,...,...,...,...,...
151,Top diseases treated with aspirin,biochirp,2,NaN,363.0,NaN,NaN,NaN
152,Top diseases associated with TP53,biochirp,2,NaN,3258.0,NaN,NaN,NaN
153,CML complete targets,biochirp,2,NaN,4916.0,NaN,NaN,NaN
154,All diseases treated with aspirin,biochirp,2,NaN,363.0,NaN,NaN,NaN


In [17]:
"""
BioChirp vs MCP Benchmark — Nature-Quality Publication Figures
================================================================
Generates a multi-panel figure suitable for Nature/Cell/Science journals.

Requirements: matplotlib, seaborn, pandas, numpy, openpyxl
Usage: python3 nature_figures.py
Output: Fig1 (main 6-panel), Fig2 (Claude vs OpenAI 2-panel), FigS1 (heatmap)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════
# NATURE JOURNAL STYLE CONFIGURATION
# ══════════════════════════════════════════════════════════════
# Nature uses Helvetica; Liberation Sans is metrically identical
FONT_FAMILY = 'Liberation Sans'
FONT_SIZE_LABEL = 7        # axis labels
FONT_SIZE_TICK = 8         # tick labels
FONT_SIZE_TITLE = 9        # panel titles
FONT_SIZE_PANEL = 10       # panel letters (a, b, c…)
FONT_SIZE_ANNOT = 6      # annotations inside plots
DPI = 300
# Nature single column = 89mm, double = 183mm, full page height ≈ 247mm
FIG_WIDTH_SINGLE = 89 / 25.4   # inches
FIG_WIDTH_DOUBLE = 183 / 25.4  # inches
FIG_WIDTH_1_5 = 136 / 25.4     # 1.5 column

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': [FONT_FAMILY, 'Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': FONT_SIZE_LABEL,
    'axes.titlesize': FONT_SIZE_TITLE,
    'axes.labelsize': FONT_SIZE_LABEL,
    'xtick.labelsize': FONT_SIZE_TICK,
    'ytick.labelsize': FONT_SIZE_TICK,
    'legend.fontsize': FONT_SIZE_TICK,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
    'xtick.minor.size': 1.5,
    'ytick.minor.size': 1.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 0.8,
    'patch.linewidth': 0.4,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'pdf.fonttype': 42,        # editable text in PDF
    'ps.fonttype': 42,
})

# ── Color palette (colorblind-safe, Nature-friendly) ──
C_BIOCHIRP = '#2ca02c'      # green
C_CLAUDE = '#1f77b4'        # blue
C_OPENAI = '#ff7f0e'        # orange
C_FAIL = '#d62728'          # red
C_PARTIAL = '#bcbd22'       # olive
C_PASS = '#2ca02c'          # green
C_GREY = '#7f7f7f'
C_LIGHT_GREY = '#c7c7c7'

MODEL_COLORS = {
    'gpt-4.1-nano': '#fdae6b',
    'gpt-4o-mini': '#fd8d3c',
    'gpt-4.1-mini': '#f16913',
    'gpt-5-nano': '#d94801',
    'gpt-5-mini': '#8c2d04',
    'Haiku 3.5': '#6baed6',
    'Sonnet 4.5': '#2171b5',
    'BioChirp': C_BIOCHIRP,
}

# ══════════════════════════════════════════════════════════════
# DATA PREPARATION
# ══════════════════════════════════════════════════════════════
# df = pd.read_excel('/mnt/user-data/uploads/result_mcp.xlsx')

# Standardize error categories
def classify_error(e):
    if pd.isna(e):
        return np.nan
    e = str(e).strip()
    if e == 'Pass':
        return 'Pass'
    elif 'Partial' in e:
        return 'Partial'
    else:
        return 'Fail'

df['Status'] = df['Error'].apply(classify_error)

# Standardize question short names
Q_MAP = {
    'CML complete targets': 'CML\n(complete)',
    'CML top targets': 'CML\n(top)',
    'Top diseases treated with aspirin': 'Aspirin\n(top)',
    'All diseases treated with aspirin': 'Aspirin\n(all)',
    'Top diseases associated with TP53': 'TP53\n(top)',
    'All diseases associated with TP53': 'TP53\n(all)',
}
Q_SHORT_FLAT = {
    'CML complete targets': 'CML (complete)',
    'CML top targets': 'CML (top)',
    'Top diseases treated with aspirin': 'Aspirin (top)',
    'All diseases treated with aspirin': 'Aspirin (all)',
    'Top diseases associated with TP53': 'TP53 (top)',
    'All diseases associated with TP53': 'TP53 (all)',
}

df['Q_short'] = df['Question'].map(Q_MAP)
df['Q_flat'] = df['Question'].map(Q_SHORT_FLAT)

QUESTIONS = list(Q_MAP.keys())

# Provider
def get_provider(m):
    if m in ['Haiku 3.5', 'Sonnet 4.5']:
        return 'Anthropic (Claude)'
    elif m == 'biochirp':
        return 'BioChirp'
    else:
        return 'OpenAI'

df['Provider'] = df['Model'].apply(get_provider)

# Separate datasets
mcp_df = df[df['Model'] != 'biochirp'].copy()
bc_df = df[df['Model'] == 'biochirp'].copy()

BIOCHIRP_COUNTS = {
    'CML complete targets': 4916,
    'CML top targets': 4916,
    'Top diseases treated with aspirin': 363,
    'All diseases treated with aspirin': 363,
    'Top diseases associated with TP53': 3258,
    'All diseases associated with TP53': 3258,
}

GPT_MODELS = ['gpt-4.1-nano', 'gpt-4o-mini', 'gpt-4.1-mini', 'gpt-5-nano', 'gpt-5-mini']
CLAUDE_MODELS = ['Haiku 3.5', 'Sonnet 4.5']
ALL_MCP_MODELS = GPT_MODELS + CLAUDE_MODELS
ALL_MODELS = ALL_MCP_MODELS + ['BioChirp']

# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
def add_panel_label(ax, label, x=-0.15, y=1.08):
    """Add bold panel letter like Nature journals."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=FONT_SIZE_PANEL, fontweight='bold',
            va='top', ha='left')

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def success_rate(sub):
    total = len(sub[sub['Status'].notna()])
    if total == 0:
        return np.nan
    passes = (sub['Status'] == 'Pass').sum()
    return passes / total * 100

# ══════════════════════════════════════════════════════════════
# FIGURE 1 — MAIN FIGURE (6 panels)
# ══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(FIG_WIDTH_DOUBLE, FIG_WIDTH_DOUBLE * 0.95))
gs = gridspec.GridSpec(3, 2, hspace=0.55, wspace=0.38,
                       left=0.08, right=0.97, top=0.95, bottom=0.06)

# ── Panel (a): Overall success rate by model ──
ax_a = fig.add_subplot(gs[0, 0])
models_ordered = ALL_MODELS
rates = []
colors_a = []
for m in models_ordered:
    if m == 'BioChirp':
        rates.append(100.0)
        colors_a.append(C_BIOCHIRP)
    else:
        sub = mcp_df[mcp_df['Model'] == m]
        rates.append(success_rate(sub))
        colors_a.append(C_CLAUDE if m in CLAUDE_MODELS else C_OPENAI)

y_pos = np.arange(len(models_ordered))
bars = ax_a.barh(y_pos, rates, height=0.65, color=colors_a, edgecolor='white',
                 linewidth=0.3, zorder=3)
ax_a.set_yticks(y_pos)
ax_a.set_yticklabels(models_ordered, fontsize=FONT_SIZE_TICK)
ax_a.set_xlabel('Success rate (%)')
ax_a.set_xlim(0, 115)
ax_a.invert_yaxis()
ax_a.axvline(x=50, color=C_LIGHT_GREY, linewidth=0.4, linestyle='--', zorder=1)

# Add percentage labels
for i, (v, m) in enumerate(zip(rates, models_ordered)):
    if not np.isnan(v):
        ax_a.text(v + 1.5, i, f'{v:.0f}%', va='center', fontsize=FONT_SIZE_ANNOT,
                  fontweight='bold' if m == 'BioChirp' else 'normal')

add_panel_label(ax_a, 'a')
ax_a.set_title('Success rate across all queries', loc='left', pad=4)

# ── Panel (b): Stacked outcome by model ──
ax_b = fig.add_subplot(gs[0, 1])
outcome_data = {'Pass': [], 'Partial': [], 'Fail': []}
for m in ALL_MCP_MODELS:
    sub = mcp_df[mcp_df['Model'] == m]
    total = len(sub[sub['Status'].notna()])
    for status in ['Pass', 'Partial', 'Fail']:
        cnt = (sub['Status'] == status).sum()
        outcome_data[status].append(cnt / total * 100 if total > 0 else 0)

x_pos = np.arange(len(ALL_MCP_MODELS))
width = 0.7
bottom = np.zeros(len(ALL_MCP_MODELS))

color_map = {'Pass': C_PASS, 'Partial': C_PARTIAL, 'Fail': C_FAIL}
for status in ['Pass', 'Partial', 'Fail']:
    ax_b.bar(x_pos, outcome_data[status], width, bottom=bottom,
             color=color_map[status], edgecolor='white', linewidth=0.2,
             label=status, zorder=3)
    bottom += np.array(outcome_data[status])

ax_b.set_xticks(x_pos)
ax_b.set_xticklabels(ALL_MCP_MODELS, rotation=45, ha='right', fontsize=FONT_SIZE_TICK - 0.5)
ax_b.set_ylabel('Proportion of runs (%)')
ax_b.set_ylim(0, 105)
ax_b.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.02),
            handlelength=1, handletextpad=0.3, columnspacing=0.8)

# Divider between OpenAI and Claude
ax_b.axvline(x=4.5, color=C_GREY, linewidth=0.5, linestyle=':', zorder=2)
ax_b.text(2, 103, 'OpenAI', ha='center', fontsize=FONT_SIZE_ANNOT, color=C_OPENAI, fontstyle='italic')
ax_b.text(5.5, 103, 'Claude', ha='center', fontsize=FONT_SIZE_ANNOT, color=C_CLAUDE, fontstyle='italic')

add_panel_label(ax_b, 'b')
ax_b.set_title('Run outcome distribution (MCP)', loc='left', pad=4)

# ── Panel (c): Data completeness — log scale grouped bars ──
ax_c = fig.add_subplot(gs[1, 0])

n_q = len(QUESTIONS)
n_m = len(ALL_MODELS)
bar_w = 0.1
x_base = np.arange(n_q)

for i, m in enumerate(ALL_MODELS):
    vals = []
    for q in QUESTIONS:
        if m == 'BioChirp':
            vals.append(BIOCHIRP_COUNTS[q])
        else:
            sub = mcp_df[(mcp_df['Model'] == m) & (mcp_df['Question'] == q) & (mcp_df['Status'] == 'Pass')]
            if len(sub) > 0:
                vals.append(sub['Count'].max())
            else:
                vals.append(0.3)  # plot just below 1 for log scale
    offset = (i - n_m / 2 + 0.5) * bar_w
    color = MODEL_COLORS.get(m, C_GREY)
    ax_c.bar(x_base + offset, vals, bar_w * 0.9, color=color,
             edgecolor='white', linewidth=0.15, zorder=3,
             label=m)

ax_c.set_yscale('log')
ax_c.set_ylabel('Records retrieved (log₁₀)')
ax_c.set_xticks(x_base)
ax_c.set_xticklabels([Q_MAP[q] for q in QUESTIONS], fontsize=FONT_SIZE_TICK - 0.5)
ax_c.set_ylim(0.2, 15000)
ax_c.axhline(y=1, color=C_LIGHT_GREY, linewidth=0.3, linestyle='-', zorder=1)

# Compact legend
handles_c = [mpatches.Patch(facecolor=MODEL_COLORS[m], label=m) for m in ALL_MODELS]
ax_c.legend(handles=handles_c, frameon=False, ncol=2, loc='upper left',
            fontsize=FONT_SIZE_ANNOT, handlelength=0.8, handletextpad=0.3,
            columnspacing=0.5, bbox_to_anchor=(0.0, 1.0))

add_panel_label(ax_c, 'c')
ax_c.set_title('Data completeness per query (best run)', loc='left', pad=4)

# ── Panel (d): Run-to-run consistency scatter ──
ax_d = fig.add_subplot(gs[1, 1])

# Diagonal reference line
ax_d.plot([0.3, 6000], [0.3, 6000], color=C_LIGHT_GREY, linewidth=0.5,
          linestyle='--', zorder=1, label='Perfect consistency')

for m in ALL_MCP_MODELS:
    x_pts, y_pts = [], []
    for q in QUESTIONS:
        r1 = mcp_df[(mcp_df['Model'] == m) & (mcp_df['Question'] == q) & (mcp_df['Run'] == 1)]
        r2 = mcp_df[(mcp_df['Model'] == m) & (mcp_df['Question'] == q) & (mcp_df['Run'] == 2)]
        if len(r1) > 0 and len(r2) > 0:
            c1 = r1['Count'].values[0] if pd.notna(r1['Count'].values[0]) else 0
            c2 = r2['Count'].values[0] if pd.notna(r2['Count'].values[0]) else 0
            x_pts.append(max(c1, 0.4))
            y_pts.append(max(c2, 0.4))
    color = MODEL_COLORS.get(m, C_GREY)
    marker = 'o' if m in GPT_MODELS else 's'
    ax_d.scatter(x_pts, y_pts, s=14, c=color, marker=marker,
                 edgecolors='white', linewidth=0.2, alpha=0.85, zorder=3, label=m)

# BioChirp points — perfect diagonal
bc_x = [BIOCHIRP_COUNTS[q] for q in QUESTIONS]
ax_d.scatter(bc_x, bc_x, s=30, c=C_BIOCHIRP, marker='D',
             edgecolors='white', linewidth=0.3, zorder=4, label='BioChirp')

ax_d.set_xscale('log')
ax_d.set_yscale('log')
ax_d.set_xlabel('Run 1 — records retrieved')
ax_d.set_ylabel('Run 2 — records retrieved')
ax_d.set_xlim(0.25, 8000)
ax_d.set_ylim(0.25, 8000)
ax_d.set_aspect('equal')

ax_d.legend(frameon=False, fontsize=FONT_SIZE_ANNOT - 0.5, loc='lower right',
            markerscale=0.8, handletextpad=0.2, labelspacing=0.3,
            ncol=2, columnspacing=0.5)

add_panel_label(ax_d, 'd')
ax_d.set_title('Reproducibility: Run 1 vs Run 2', loc='left', pad=4)

# ── Panel (e): Error type breakdown (MCP only) ──
ax_e = fig.add_subplot(gs[2, 0])

error_types_raw = mcp_df[mcp_df['Error'].notna() & (mcp_df['Error'] != 'Pass') &
                          (~mcp_df['Error'].str.contains('Partial', na=False))]['Error'].str.strip()

error_mapping = {}
for e in error_types_raw.unique():
    if 'Upstream' in str(e) and 'Availability' in str(e):
        error_mapping[e] = 'Upstream Service\nUnavailable'
    elif 'Upstream' in str(e):
        error_mapping[e] = 'Upstream Retrieval\nFailure'
    elif 'MaxTurns' in str(e):
        error_mapping[e] = 'Max Turns\nExceeded'
    elif 'BadRequest' in str(e):
        error_mapping[e] = 'Context Window\nOverflow'
    elif 'Timeout' in str(e):
        error_mapping[e] = 'MCP Tool\nTimeout'
    elif 'Unclassified' in str(e):
        error_mapping[e] = 'Unclassified'
    else:
        error_mapping[e] = str(e)

mcp_errors = mcp_df[(mcp_df['Status'] == 'Fail')].copy()
mcp_errors['ErrorType'] = mcp_errors['Error'].str.strip().map(error_mapping)

error_counts_by_provider = mcp_errors.groupby(['Provider', 'ErrorType']).size().unstack(fill_value=0)

error_order = ['Upstream Retrieval\nFailure', 'Max Turns\nExceeded', 'Context Window\nOverflow',
               'MCP Tool\nTimeout', 'Upstream Service\nUnavailable', 'Unclassified']
error_order = [e for e in error_order if e in error_counts_by_provider.columns]

x_err = np.arange(len(error_order))
width_e = 0.35

for i, prov in enumerate(['OpenAI', 'Anthropic (Claude)']):
    if prov in error_counts_by_provider.index:
        vals = [error_counts_by_provider.loc[prov, e] if e in error_counts_by_provider.columns else 0
                for e in error_order]
    else:
        vals = [0] * len(error_order)
    color = C_OPENAI if prov == 'OpenAI' else C_CLAUDE
    offset = -width_e / 2 + i * width_e
    bars_e = ax_e.bar(x_err + offset, vals, width_e * 0.9, color=color,
                      edgecolor='white', linewidth=0.2, zorder=3,
                      label=prov)
    for bar, v in zip(bars_e, vals):
        if v > 0:
            ax_e.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                      str(v), ha='center', va='bottom', fontsize=FONT_SIZE_ANNOT,
                      fontweight='bold')

ax_e.set_xticks(x_err)
ax_e.set_xticklabels(error_order, fontsize=FONT_SIZE_TICK - 1)
ax_e.set_ylabel('Number of failed runs')
ax_e.legend(frameon=False, loc='upper right', fontsize=FONT_SIZE_TICK)

# BioChirp annotation
ax_e.annotate('BioChirp: 0 errors\nacross all runs',
              xy=(0.98, 0.65), xycoords='axes fraction',
              fontsize=FONT_SIZE_ANNOT, color=C_BIOCHIRP,
              fontweight='bold', ha='right',
              bbox=dict(boxstyle='round,pad=0.3', facecolor='#e8f5e9',
                        edgecolor=C_BIOCHIRP, linewidth=0.5, alpha=0.9))

add_panel_label(ax_e, 'e')
ax_e.set_title('MCP failure modes by provider', loc='left', pad=4)

# ── Panel (f): Latency boxplot (OpenAI models only, since Claude Desktop has no latency) ──
ax_f = fig.add_subplot(gs[2, 1])

lat_data = []
lat_labels = []
lat_colors = []
for m in GPT_MODELS:
    sub = mcp_df[(mcp_df['Model'] == m) & (mcp_df['Latency'].notna())]
    if len(sub) > 0:
        lat_data.append(sub['Latency'].values)
        lat_labels.append(m)
        lat_colors.append(MODEL_COLORS[m])

bp = ax_f.boxplot(lat_data, patch_artist=True, widths=0.55,
                  medianprops=dict(color='black', linewidth=0.8),
                  whiskerprops=dict(linewidth=0.5),
                  capprops=dict(linewidth=0.5),
                  flierprops=dict(marker='o', markersize=2, markerfacecolor=C_GREY,
                                  markeredgewidth=0.2),
                  zorder=3)

for patch, color in zip(bp['boxes'], lat_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
    patch.set_edgecolor('white')
    patch.set_linewidth(0.3)

ax_f.set_xticklabels(lat_labels, rotation=30, ha='right', fontsize=FONT_SIZE_TICK)
ax_f.set_ylabel('Latency (seconds)')

# Add markers for successful vs failed runs
for i, m in enumerate(lat_labels):
    sub = mcp_df[(mcp_df['Model'] == m) & (mcp_df['Latency'].notna())]
    success = sub[sub['Status'] == 'Pass']['Latency'].values
    fail = sub[sub['Status'] != 'Pass']['Latency'].values
    jitter_s = np.random.uniform(-0.12, 0.12, len(success))
    jitter_f = np.random.uniform(-0.12, 0.12, len(fail))
    ax_f.scatter(np.full(len(success), i + 1) + jitter_s, success,
                 s=8, c=C_PASS, marker='o', edgecolors='white',
                 linewidth=0.2, alpha=0.8, zorder=4)
    ax_f.scatter(np.full(len(fail), i + 1) + jitter_f, fail,
                 s=8, c=C_FAIL, marker='x', linewidth=0.4, alpha=0.6, zorder=4)

# Legend for markers
leg_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_PASS, markersize=4, label='Pass'),
    Line2D([0], [0], marker='x', color=C_FAIL, markersize=4, markeredgewidth=0.6, label='Fail', linestyle=''),
]
ax_f.legend(handles=leg_handles, frameon=False, loc='upper left', fontsize=FONT_SIZE_TICK)

add_panel_label(ax_f, 'f')
ax_f.set_title('Query latency by model (OpenAI MCP)', loc='left', pad=4)

fig.savefig('Fig1_main.png', dpi=DPI, facecolor='white')
fig.savefig('Fig1_main.pdf', dpi=DPI, facecolor='white')
plt.show()
print("✓ Figure 1 saved")
# plt.close(fig)


# ══════════════════════════════════════════════════════════════
# FIGURE 2 — CLAUDE vs OPENAI DEEP DIVE (2 panels)
# ══════════════════════════════════════════════════════════════
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(FIG_WIDTH_DOUBLE, FIG_WIDTH_DOUBLE * 0.32))
plt.subplots_adjust(wspace=0.35, left=0.07, right=0.97, top=0.85, bottom=0.18)

# ── Panel (a): Per-question success rate — Claude vs OpenAI ──
claude_rates = []
openai_rates = []
for q in QUESTIONS:
    c_sub = mcp_df[(mcp_df['Provider'] == 'Anthropic (Claude)') & (mcp_df['Question'] == q)]
    o_sub = mcp_df[(mcp_df['Provider'] == 'OpenAI') & (mcp_df['Question'] == q)]
    claude_rates.append(success_rate(c_sub))
    openai_rates.append(success_rate(o_sub))

x2 = np.arange(len(QUESTIONS))
w2 = 0.35

bars_c = ax2a.bar(x2 - w2 / 2, claude_rates, w2, color=C_CLAUDE,
                  edgecolor='white', linewidth=0.2, label='Anthropic (Claude)', zorder=3)
bars_o = ax2a.bar(x2 + w2 / 2, openai_rates, w2, color=C_OPENAI,
                  edgecolor='white', linewidth=0.2, label='OpenAI', zorder=3)

for bars in [bars_c, bars_o]:
    for bar in bars:
        h = bar.get_height()
        if h > 0 and not np.isnan(h):
            ax2a.text(bar.get_x() + bar.get_width() / 2, h + 1.5,
                      f'{h:.0f}', ha='center', va='bottom',
                      fontsize=FONT_SIZE_ANNOT, fontweight='bold')

ax2a.set_xticks(x2)
ax2a.set_xticklabels([Q_MAP[q] for q in QUESTIONS], fontsize=FONT_SIZE_TICK - 0.5)
ax2a.set_ylabel('Success rate (%)')
ax2a.set_ylim(0, 115)
ax2a.axhline(100, color=C_BIOCHIRP, linewidth=0.6, linestyle='--', zorder=1, alpha=0.5)
ax2a.text(0.02, 92, 'BioChirp baseline (100%)', fontsize=FONT_SIZE_ANNOT,
          color=C_BIOCHIRP, ha='left', fontstyle='italic', fontweight='bold')
ax2a.legend(frameon=False, loc='upper right', fontsize=FONT_SIZE_TICK,
            bbox_to_anchor=(1.0, 0.88))

add_panel_label(ax2a, 'a', x=-0.1)
ax2a.set_title('MCP success rate by provider and query', loc='left', pad=4)

# ── Panel (b): Per-model success (grouped by provider with separation) ──
model_order_fig2 = ['gpt-4.1-nano', 'gpt-4o-mini', 'gpt-4.1-mini', 'gpt-5-nano', 'gpt-5-mini',
                    'Haiku 3.5', 'Sonnet 4.5']
rates_fig2 = []
colors_fig2 = []
for m in model_order_fig2:
    sub = mcp_df[mcp_df['Model'] == m]
    rates_fig2.append(success_rate(sub))
    colors_fig2.append(C_CLAUDE if m in CLAUDE_MODELS else C_OPENAI)

y2 = np.arange(len(model_order_fig2))
bars2 = ax2b.barh(y2, rates_fig2, height=0.6, color=colors_fig2,
                   edgecolor='white', linewidth=0.2, zorder=3)

ax2b.set_yticks(y2)
ax2b.set_yticklabels(model_order_fig2, fontsize=FONT_SIZE_TICK)
ax2b.set_xlabel('Success rate (%)')
ax2b.set_xlim(0, 130)
ax2b.invert_yaxis()

# Divider
ax2b.axhline(y=4.5, color=C_GREY, linewidth=0.5, linestyle=':', zorder=2)

for i, v in enumerate(rates_fig2):
    if not np.isnan(v):
        ax2b.text(v + 1.5, i, f'{v:.0f}%', va='center', fontsize=FONT_SIZE_ANNOT, fontweight='bold')

# Provider mean annotations — place in right margin area
mean_openai = np.nanmean([r for r, m in zip(rates_fig2, model_order_fig2) if m in GPT_MODELS])
mean_claude = np.nanmean([r for r, m in zip(rates_fig2, model_order_fig2) if m in CLAUDE_MODELS])

ax2b.annotate('', xy=(118, -0.3), xytext=(118, 4.3),
              arrowprops=dict(arrowstyle='-[', color=C_OPENAI, linewidth=0.6, mutation_scale=5))
ax2b.text(120, 2, f'OpenAI\nμ={mean_openai:.0f}%', fontsize=FONT_SIZE_ANNOT, color=C_OPENAI,
          va='center', ha='left', linespacing=1.3)

ax2b.annotate('', xy=(118, 4.7), xytext=(118, 6.3),
              arrowprops=dict(arrowstyle='-[', color=C_CLAUDE, linewidth=0.6, mutation_scale=5))
ax2b.text(120, 5.5, f'Claude\nμ={mean_claude:.0f}%', fontsize=FONT_SIZE_ANNOT, color=C_CLAUDE,
          va='center', ha='left', linespacing=1.3)

add_panel_label(ax2b, 'b', x=-0.12)
ax2b.set_title('MCP success rate by individual model', loc='left', pad=4)

fig2.savefig('Fig2_provider_comparison.png', dpi=DPI, facecolor='white')
fig2.savefig('Fig2_provider_comparison.pdf', dpi=DPI, facecolor='white')
print("✓ Figure 2 saved")
plt.close(fig2)


# ══════════════════════════════════════════════════════════════
# FIGURE S1 — SUPPLEMENTARY HEATMAP
# ══════════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(figsize=(FIG_WIDTH_DOUBLE, FIG_WIDTH_DOUBLE * 0.38))

# Build matrix: rows = questions, cols = models, values = best count (or -1 for fail)
heatmap_models = ALL_MCP_MODELS + ['BioChirp']
matrix = np.full((len(QUESTIONS), len(heatmap_models)), np.nan)
annot_matrix = np.empty((len(QUESTIONS), len(heatmap_models)), dtype=object)

for j, m in enumerate(heatmap_models):
    for i, q in enumerate(QUESTIONS):
        if m == 'BioChirp':
            matrix[i, j] = BIOCHIRP_COUNTS[q]
            annot_matrix[i, j] = f'{BIOCHIRP_COUNTS[q]:,}'
        else:
            sub = mcp_df[(mcp_df['Model'] == m) & (mcp_df['Question'] == q)]
            passes = sub[sub['Status'] == 'Pass']
            partials = sub[sub['Status'] == 'Partial']
            if len(passes) > 0:
                best = passes['Count'].max()
                matrix[i, j] = best
                annot_matrix[i, j] = f'{int(best)}'
            elif len(partials) > 0:
                matrix[i, j] = 0.5
                annot_matrix[i, j] = '~1'
            else:
                matrix[i, j] = 0
                annot_matrix[i, j] = '✗'

# Custom colormap: fail=red, low=yellow, high=green
cmap = LinearSegmentedColormap.from_list('rg',
    [(0, '#fee0d2'), (0.001, '#fee0d2'), (0.01, '#ffffb2'),
     (0.1, '#addd8e'), (0.5, '#41ab5d'), (1.0, '#004529')])

# Log-normalize for better color spread
log_matrix = np.log10(matrix + 1)
log_max = np.log10(max(BIOCHIRP_COUNTS.values()) + 1)

im = ax3.imshow(log_matrix / log_max, cmap=cmap, aspect='auto', vmin=0, vmax=1)

# Annotations
for i in range(len(QUESTIONS)):
    for j in range(len(heatmap_models)):
        txt = annot_matrix[i, j]
        color = 'white' if log_matrix[i, j] / log_max > 0.45 else 'black'
        if txt == '✗':
            color = '#d62728'
        weight = 'bold' if heatmap_models[j] == 'BioChirp' else 'normal'
        ax3.text(j, i, txt, ha='center', va='center',
                 fontsize=FONT_SIZE_ANNOT, color=color, fontweight=weight)

ax3.set_xticks(np.arange(len(heatmap_models)))
ax3.set_xticklabels(heatmap_models, rotation=45, ha='right', fontsize=FONT_SIZE_TICK)
ax3.set_yticks(np.arange(len(QUESTIONS)))
ax3.set_yticklabels([Q_SHORT_FLAT[q] for q in QUESTIONS], fontsize=FONT_SIZE_TICK)

# Dividers
ax3.axvline(x=4.5, color='white', linewidth=1.5)   # OpenAI | Claude
ax3.axvline(x=6.5, color='white', linewidth=1.5)    # Claude | BioChirp

# Provider labels on top
ax3.text(2, -0.8, 'OpenAI', ha='center', fontsize=FONT_SIZE_TICK, color=C_OPENAI, fontweight='bold')
ax3.text(5.5, -0.8, 'Claude', ha='center', fontsize=FONT_SIZE_TICK, color=C_CLAUDE, fontweight='bold')
ax3.text(7, -0.8, 'BioChirp', ha='center', fontsize=FONT_SIZE_TICK, color=C_BIOCHIRP, fontweight='bold')

ax3.set_title('Best records retrieved per query–model pair (✗ = all runs failed)',
              loc='left', fontsize=FONT_SIZE_TITLE, pad=14)

plt.tight_layout()
fig3.savefig('FigS1_heatmap.png', dpi=DPI, facecolor='white')
fig3.savefig('FigS1_heatmap.pdf', dpi=DPI, facecolor='white')
print("✓ Figure S1 saved")
plt.close(fig3)

print("\n═══ All figures generated successfully ═══")
print("  Fig1_main.png/pdf         — 6-panel main figure")
print("  Fig2_provider_comparison   — Claude vs OpenAI deep dive")
print("  FigS1_heatmap             — Supplementary heatmap")

✓ Figure 1 saved
✓ Figure 2 saved
✓ Figure S1 saved

═══ All figures generated successfully ═══
  Fig1_main.png/pdf         — 6-panel main figure
  Fig2_provider_comparison   — Claude vs OpenAI deep dive
  FigS1_heatmap             — Supplementary heatmap


In [ ]:
df

In [16]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd

# ?? Data ??????????????????????????????????????????????????????????
queries = ["CML (complete)", "CML (top)", "Aspirin (top)", "Aspirin (all)", "TP53 (top)", "TP53 (all)"]
openai_models = ["gpt-4.1-nano", "gpt-4o-mini", "gpt-4.1-mini", "gpt-5-nano", "gpt-5-mini"]
claude_models = ["Haiku 3.5", "Sonnet 4.5"]

biochirp_vals = [4916, 4916, 363, 363, 3258, 3258]

# Best-of-2-runs data: None = all failed, negative = partial
openai_data = np.array([
    [None,  None,  None,  10,   374],   # CML complete
    [-1,    None,  None,  20,   10],     # CML top
    [None,  10,    29,    18,   168],    # Aspirin top
    [None,  None,  168,   168,  168],    # Aspirin all
    [None,  None,  10,    25,   None],   # TP53 top
    [-1,    25,    46,    20,   None],   # TP53 all
], dtype=object)

claude_data = np.array([
    [5,   99],   # CML complete
    [5,   20],   # CML top
    [27,  55],   # Aspirin top
    [41,  99],   # Aspirin all
    [5,   25],   # TP53 top
    [20,  99],   # TP53 all
], dtype=object)

# ?? Style Setup ???????????????????????????????????????????????????
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor': '#0a0e1a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
})

fig, ax = plt.subplots(figsize=(18, 8))
fig.set_facecolor('#0a0e1a')
ax.set_facecolor('#0a0e1a')
ax.axis('off')

n_rows = len(queries)
n_openai = len(openai_models)
n_claude = len(claude_models)
gap = 0.4  # gap between provider groups
bc_gap = 0.4

cell_w, cell_h = 1.0, 0.75
pad = 0.04

# X positions
x_openai = np.arange(n_openai) * (cell_w + pad)
x_claude = x_openai[-1] + cell_w + gap + np.arange(n_claude) * (cell_w + pad)
x_bc = x_claude[-1] + cell_w + bc_gap

total_w = x_bc + cell_w
y_positions = np.arange(n_rows) * (cell_h + pad)
y_positions = y_positions[::-1]  # top to bottom

# ?? Color functions ???????????????????????????????????????????????
def fail_color():
    return '#1a0808'

def fail_border():
    return '#5c1a1a'

def partial_color():
    return '#1a1205'

def partial_border():
    return '#6b4c0a'

def pass_color(val, max_val=500):
    t = min(val / max_val, 1.0)
    r = 0.04 + 0.02 * t
    g = 0.12 + 0.38 * t
    b = 0.06 + 0.12 * t
    return (r, g, b)

def pass_border(val, max_val=500):
    t = min(val / max_val, 1.0)
    return (0.1 + 0.1*t, 0.45 + 0.35*t, 0.15 + 0.2*t)

def biochirp_color():
    return (0.05, 0.32, 0.15)

def biochirp_border():
    return (0.13, 0.77, 0.37)

def text_color_for_val(val, is_fail=False, is_partial=False, is_biochirp=False):
    if is_fail: return '#ef4444'
    if is_partial: return '#fbbf24'
    if is_biochirp: return '#ffffff'
    if val is not None and val > 100: return '#ffffff'
    return '#86efac'

# ?? Draw cells ????????????????????????????????????????????????????
def draw_cell(ax, x, y, w, h, facecolor, edgecolor, text, textcolor, 
              fontsize=12, fontweight='bold', is_fail=False, is_biochirp=False, glow=False):
    
    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02",
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=1.2 if not is_biochirp else 1.8,
        zorder=2
    )
    ax.add_patch(rect)

    # Fail hatching
    if is_fail:
        for i in range(int(w / 0.15) + 5):
            x0 = x + i * 0.15
            ax.plot([x0, x0 - h*0.6], [y, y + h], 
                    color='#3d1515', linewidth=0.5, alpha=0.5, zorder=3,
                    clip_on=True, clip_path=rect)

    # BioChirp glow
    if glow:
        glow_rect = FancyBboxPatch(
            (x - 0.02, y - 0.02), w + 0.04, h + 0.04,
            boxstyle="round,pad=0.03",
            facecolor='none',
            edgecolor=(0.13, 0.77, 0.37, 0.25),
            linewidth=3,
            zorder=1
        )
        ax.add_patch(glow_rect)

    ax.text(x + w/2, y + h/2, text,
            ha='center', va='center',
            fontsize=fontsize, fontweight=fontweight,
            color=textcolor, zorder=4)

# ?? Draw the grid ?????????????????????????????????????????????????

# Provider labels
mid_openai = (x_openai[0] + x_openai[-1] + cell_w) / 2
mid_claude = (x_claude[0] + x_claude[-1] + cell_w) / 2
mid_bc = x_bc + cell_w / 2
top_y = y_positions[0] + cell_h + 0.55

ax.text(mid_openai, top_y, 'OpenAI', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#f59e0b', zorder=5)
ax.text(mid_claude, top_y, 'Claude', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#06b6d4', zorder=5)
ax.text(mid_bc, top_y, 'BioChirp', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#22c55e', zorder=5)

# Separator lines
sep_x1 = x_openai[-1] + cell_w + gap/2
sep_x2 = x_claude[-1] + cell_w + bc_gap/2
for sx in [sep_x1, sep_x2]:
    ax.plot([sx, sx], [y_positions[-1] - 0.1, y_positions[0] + cell_h + 0.35],
            color='#1e293b', linewidth=1.2, linestyle='--', alpha=0.6, zorder=1)

# Model headers
header_y = y_positions[0] + cell_h + 0.18
for i, m in enumerate(openai_models):
    ax.text(x_openai[i] + cell_w/2, header_y, m, ha='center', va='center',
            fontsize=8.5, fontweight='semibold', color='#b08030', rotation=25, zorder=5)

for i, m in enumerate(claude_models):
    ax.text(x_claude[i] + cell_w/2, header_y, m, ha='center', va='center',
            fontsize=8.5, fontweight='semibold', color='#0ea5c7', rotation=25, zorder=5)

ax.text(x_bc + cell_w/2, header_y, 'BioChirp', ha='center', va='center',
        fontsize=8.5, fontweight='bold', color='#22c55e', rotation=25, zorder=5)

# Row labels
for i, q in enumerate(queries):
    ax.text(x_openai[0] - 0.15, y_positions[i] + cell_h/2, q,
            ha='right', va='center', fontsize=11, fontweight='semibold',
            color='#cbd5e1', zorder=5)

# OpenAI cells
for r in range(n_rows):
    for c in range(n_openai):
        val = openai_data[r][c]
        x = x_openai[c]
        y = y_positions[r]
        
        if val is None:
            draw_cell(ax, x, y, cell_w, cell_h,
                      fail_color(), fail_border(), '?', '#ef4444',
                      fontsize=16, is_fail=True)
        elif val < 0:
            draw_cell(ax, x, y, cell_w, cell_h,
                      partial_color(), partial_border(), f'~{abs(val)}', '#fbbf24',
                      fontsize=11)
        else:
            draw_cell(ax, x, y, cell_w, cell_h,
                      pass_color(val), pass_border(val), 
                      f'{val:,}' if val >= 1000 else str(val),
                      text_color_for_val(val), fontsize=12)

# Claude cells
for r in range(n_rows):
    for c in range(n_claude):
        val = claude_data[r][c]
        x = x_claude[c]
        y = y_positions[r]
        draw_cell(ax, x, y, cell_w, cell_h,
                  pass_color(val), pass_border(val),
                  f'{val:,}' if val >= 1000 else str(val),
                  text_color_for_val(val), fontsize=12)

# BioChirp cells
for r in range(n_rows):
    val = biochirp_vals[r]
    draw_cell(ax, x_bc, y_positions[r], cell_w, cell_h,
              biochirp_color(), biochirp_border(),
              f'{val:,}', '#ffffff',
              fontsize=13, fontweight='bold', is_biochirp=True, glow=True)

# ?? Title ?????????????????????????????????????????????????????????
ax.text((x_openai[0] + x_bc + cell_w) / 2, top_y + 0.52,
        'Best Records Retrieved per Query?Model Pair',
        ha='center', va='center', fontsize=17, fontweight='bold',
        color='#e2e8f0', zorder=5)

ax.text((x_openai[0] + x_bc + cell_w) / 2, top_y + 0.28,
        'Best of 2 runs  ·  ? = all runs failed  ·  ~ = partial success',
        ha='center', va='center', fontsize=9, color='#64748b', zorder=5,
        fontstyle='italic')

# ?? Legend ?????????????????????????????????????????????????????????
legend_y = y_positions[-1] - 0.55
legend_items = [
    (fail_color(), fail_border(), '? All Failed', '#ef4444'),
    (partial_color(), partial_border(), '~ Partial', '#fbbf24'),
    (pass_color(20), pass_border(20), 'Low Count', '#86efac'),
    (pass_color(200), pass_border(200), 'Medium', '#4ade80'),
    (pass_color(500), pass_border(500), 'High', '#ffffff'),
    (biochirp_color(), biochirp_border(), 'BioChirp', '#22c55e'),
]

legend_x_start = x_openai[0]
for idx, (fc, ec, label, tc) in enumerate(legend_items):
    lx = legend_x_start + idx * 1.6
    rect = FancyBboxPatch((lx, legend_y), 0.22, 0.22,
                          boxstyle="round,pad=0.01",
                          facecolor=fc, edgecolor=ec, linewidth=1, zorder=2)
    ax.add_patch(rect)
    ax.text(lx + 0.35, legend_y + 0.11, label,
            ha='left', va='center', fontsize=8.5, color=tc, zorder=5)

# ?? Final layout ??????????????????????????????????????????????????
ax.set_xlim(x_openai[0] - 2.2, x_bc + cell_w + 0.3)
ax.set_ylim(y_positions[-1] - 0.85, top_y + 0.75)
ax.set_aspect('equal')

plt.tight_layout(pad=0.5)
plt.savefig('biochirp_heatmap.png', dpi=200, 
            bbox_inches='tight', facecolor='#0a0e1a', edgecolor='none')
plt.savefig('biochirp_heatmap.pdf', 
            bbox_inches='tight', facecolor='#0a0e1a', edgecolor='none')
print("Saved heatmap to biochirp_heatmap.png and .pdf")

Saved heatmap to biochirp_heatmap.png and .pdf


In [1]:
"""
BioChirp vs MCP — Nature-Quality A4 Multi-Panel Figure
=======================================================
Single A4 page, 8 panels (a–h), white background, Liberation Sans 7pt.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════════════════
df = pd.read_excel('result_mcp.xlsx')

def classify_error(e):
    if pd.isna(e): return np.nan
    e = str(e).strip()
    if e == 'Pass': return 'Pass'
    elif 'Partial' in e: return 'Partial'
    else: return 'Fail'

df['Status'] = df['Error'].apply(classify_error)

Q_MAP = {
    'CML complete targets': 'CML\n(complete)',
    'CML top targets': 'CML\n(top)',
    'Top diseases treated with aspirin': 'Aspirin\n(top)',
    'All diseases treated with aspirin': 'Aspirin\n(all)',
    'Top diseases associated with TP53': 'TP53\n(top)',
    'All diseases associated with TP53': 'TP53\n(all)',
}
Q_SHORT = {
    'CML complete targets': 'CML (complete)',
    'CML top targets': 'CML (top)',
    'Top diseases treated with aspirin': 'Aspirin (top)',
    'All diseases treated with aspirin': 'Aspirin (all)',
    'Top diseases associated with TP53': 'TP53 (top)',
    'All diseases associated with TP53': 'TP53 (all)',
}
QUESTIONS = list(Q_MAP.keys())

def get_provider(m):
    if m in ['Haiku 3.5', 'Sonnet 4.5']: return 'Claude'
    elif m == 'biochirp': return 'BioChirp'
    else: return 'OpenAI'

df['Provider'] = df['Model'].apply(get_provider)
mcp_df = df[df['Model'] != 'biochirp'].copy()

BIOCHIRP = {
    'CML complete targets': 4916, 'CML top targets': 4916,
    'Top diseases treated with aspirin': 363, 'All diseases treated with aspirin': 363,
    'Top diseases associated with TP53': 3258, 'All diseases associated with TP53': 3258,
}
GPT = ['gpt-4.1-nano','gpt-4o-mini','gpt-4.1-mini','gpt-5-nano','gpt-5-mini']
CLAUDE = ['Haiku 3.5','Sonnet 4.5']
ALL_MCP = GPT + CLAUDE

def success_rate(sub):
    t = len(sub[sub['Status'].notna()])
    return (sub['Status'] == 'Pass').sum() / t * 100 if t > 0 else np.nan

# ═══════════════════════════════════════════════════
# STYLE — Nature journal, A4, white, 7pt
# ═══════════════════════════════════════════════════
FS = 7  # base font
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Liberation Sans', 'Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': FS,
    'axes.titlesize': FS + 1,
    'axes.labelsize': FS,
    'xtick.labelsize': FS - 0.5,
    'ytick.labelsize': FS - 0.5,
    'legend.fontsize': FS - 1,
    'axes.linewidth': 0.4,
    'xtick.major.width': 0.4,
    'ytick.major.width': 0.4,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 0.7,
    'patch.linewidth': 0.3,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.04,
    'pdf.fonttype': 42,
})

# Colors — muted, print-friendly, colorblind-safe
C_BC  = '#2ca02c'
C_CL  = '#1f77b4'
C_OA  = '#e6850e'
C_FAIL = '#c0392b'
C_PART = '#b8a920'
C_PASS = '#27ae60'
C_GY  = '#888888'
C_LG  = '#cccccc'

MC = {
    'gpt-4.1-nano':'#fdbe85','gpt-4o-mini':'#fd8d3c','gpt-4.1-mini':'#e6550d',
    'gpt-5-nano':'#a63603','gpt-5-mini':'#7f2704',
    'Haiku 3.5':'#9ecae1','Sonnet 4.5':'#3182bd','BioChirp':C_BC,
}

def panel_label(ax, lbl, x=-0.14, y=1.07):
    ax.text(x, y, lbl, transform=ax.transAxes, fontsize=FS+2,
            fontweight='bold', va='top', ha='left')

# ═══════════════════════════════════════════════════
# A4 FIGURE — 210 × 297 mm
# ═══════════════════════════════════════════════════
A4_W = 210 / 25.4  # 8.27 in
A4_H = 297 / 25.4  # 11.69 in

fig = plt.figure(figsize=(A4_W, A4_H), facecolor='white')

# Layout: 4 rows × 2 cols, row 4 = heatmap spanning full width
outer = gridspec.GridSpec(4, 2, figure=fig,
    hspace=0.52, wspace=0.38,
    left=0.09, right=0.96, top=0.965, bottom=0.035,
    height_ratios=[1, 1, 1, 1.15])

# ───────────────────────────────────────────────────
# (a) Success rate by model — include BioChirp
# ───────────────────────────────────────────────────
ax_a = fig.add_subplot(outer[0, 0])
models_a = ALL_MCP + ['BioChirp']
rates_a, colors_a = [], []
for m in models_a:
    if m == 'BioChirp':
        rates_a.append(100.0); colors_a.append(C_BC)
    else:
        rates_a.append(success_rate(mcp_df[mcp_df['Model']==m]))
        colors_a.append(C_CL if m in CLAUDE else C_OA)

y_a = np.arange(len(models_a))
ax_a.barh(y_a, rates_a, height=0.62, color=colors_a, edgecolor='white', linewidth=0.25, zorder=3)
ax_a.set_yticks(y_a)
ax_a.set_yticklabels(models_a)
ax_a.set_xlabel('Success rate (%)')
ax_a.set_xlim(0, 118)
ax_a.invert_yaxis()
ax_a.axvline(50, color=C_LG, lw=0.35, ls='--', zorder=1)
for i, v in enumerate(rates_a):
    if not np.isnan(v):
        w = 'bold' if models_a[i]=='BioChirp' else 'normal'
        ax_a.text(v+1.2, i, f'{v:.0f}%', va='center', fontsize=FS-1.5, fontweight=w)
# Provider brackets
ax_a.axhline(4.5, color=C_GY, lw=0.4, ls=':', zorder=1)
ax_a.axhline(6.5, color=C_GY, lw=0.4, ls=':', zorder=1)
panel_label(ax_a, 'a')
ax_a.set_title('Success rate across all queries', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (b) Stacked outcome — MCP models
# ───────────────────────────────────────────────────
ax_b = fig.add_subplot(outer[0, 1])
od = {'Pass':[], 'Partial':[], 'Fail':[]}
for m in ALL_MCP:
    sub = mcp_df[mcp_df['Model']==m]
    t = len(sub[sub['Status'].notna()])
    for s in ['Pass','Partial','Fail']:
        od[s].append((sub['Status']==s).sum()/t*100 if t>0 else 0)

x_b = np.arange(len(ALL_MCP))
bot = np.zeros(len(ALL_MCP))
cm = {'Pass':C_PASS, 'Partial':C_PART, 'Fail':C_FAIL}
for s in ['Pass','Partial','Fail']:
    ax_b.bar(x_b, od[s], 0.68, bottom=bot, color=cm[s], edgecolor='white', lw=0.2, label=s, zorder=3)
    bot += np.array(od[s])

ax_b.set_xticks(x_b)
ax_b.set_xticklabels(ALL_MCP, rotation=40, ha='right')
ax_b.set_ylabel('Proportion of runs (%)')
ax_b.set_ylim(0, 108)
ax_b.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5,1.02),
            handlelength=0.8, handletextpad=0.25, columnspacing=0.6)
ax_b.axvline(4.5, color=C_GY, lw=0.4, ls=':', zorder=2)
ax_b.text(2, 104, 'OpenAI', ha='center', fontsize=FS-1.5, color=C_OA, fontstyle='italic')
ax_b.text(5.5, 104, 'Claude', ha='center', fontsize=FS-1.5, color=C_CL, fontstyle='italic')
panel_label(ax_b, 'b')
ax_b.set_title('Run outcome distribution (MCP)', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (c) Claude vs OpenAI per-question — with BioChirp baseline
# ───────────────────────────────────────────────────
ax_c = fig.add_subplot(outer[1, 0])
cr, oar = [], []
for q in QUESTIONS:
    cr.append(success_rate(mcp_df[(mcp_df['Provider']=='Claude')&(mcp_df['Question']==q)]))
    oar.append(success_rate(mcp_df[(mcp_df['Provider']=='OpenAI')&(mcp_df['Question']==q)]))

x_c = np.arange(len(QUESTIONS))
w_c = 0.32
ax_c.bar(x_c - w_c, oar, w_c, color=C_OA, edgecolor='white', lw=0.2, label='OpenAI', zorder=3)
ax_c.bar(x_c, cr, w_c, color=C_CL, edgecolor='white', lw=0.2, label='Claude', zorder=3)
ax_c.bar(x_c + w_c, [100]*len(QUESTIONS), w_c, color=C_BC, edgecolor='white', lw=0.2, label='BioChirp', zorder=3, alpha=0.7)

for bars_data, offset in [(oar, -w_c), (cr, 0)]:
    for i, v in enumerate(bars_data):
        if v > 0 and not np.isnan(v):
            ax_c.text(i + offset + w_c/2, v + 1.5, f'{v:.0f}', ha='center', fontsize=FS-2, fontweight='bold')

ax_c.set_xticks(x_c)
ax_c.set_xticklabels([Q_MAP[q] for q in QUESTIONS])
ax_c.set_ylabel('Success rate (%)')
ax_c.set_ylim(0, 118)
ax_c.legend(frameon=False, ncol=3, loc='upper right', fontsize=FS-1.5,
            handlelength=0.8, handletextpad=0.25, columnspacing=0.5)
panel_label(ax_c, 'c')
ax_c.set_title('Provider success rate per query', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (d) Data completeness — grouped bars log scale
# ───────────────────────────────────────────────────
ax_d = fig.add_subplot(outer[1, 1])
ALL_MODELS = ALL_MCP + ['BioChirp']
n_q = len(QUESTIONS); n_m = len(ALL_MODELS)
bw = 0.8 / n_m

for i, m in enumerate(ALL_MODELS):
    vals = []
    for q in QUESTIONS:
        if m == 'BioChirp':
            vals.append(BIOCHIRP[q])
        else:
            sub = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)&(mcp_df['Status']=='Pass')]
            vals.append(sub['Count'].max() if len(sub)>0 else 0.3)
    offset = (i - n_m/2 + 0.5) * bw
    ax_d.bar(np.arange(n_q) + offset, vals, bw*0.88, color=MC.get(m, C_GY),
             edgecolor='white', lw=0.12, zorder=3, label=m)

ax_d.set_yscale('log')
ax_d.set_ylabel('Records retrieved (log₁₀)')
ax_d.set_xticks(np.arange(n_q))
ax_d.set_xticklabels([Q_MAP[q] for q in QUESTIONS])
ax_d.set_ylim(0.2, 12000)
handles_d = [mpatches.Patch(facecolor=MC[m], label=m) for m in ALL_MODELS]
ax_d.legend(handles=handles_d, frameon=False, ncol=2, loc='upper left',
            fontsize=FS-2, handlelength=0.6, handletextpad=0.2, columnspacing=0.4)
panel_label(ax_d, 'd')
ax_d.set_title('Data completeness per query (best run)', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (e) Reproducibility scatter — Run1 vs Run2
# ───────────────────────────────────────────────────
ax_e = fig.add_subplot(outer[2, 0])
ax_e.plot([0.3, 6000], [0.3, 6000], color=C_LG, lw=0.4, ls='--', zorder=1)

for m in ALL_MCP:
    xp, yp = [], []
    for q in QUESTIONS:
        r1 = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)&(mcp_df['Run']==1)]
        r2 = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)&(mcp_df['Run']==2)]
        if len(r1)>0 and len(r2)>0:
            c1 = r1['Count'].values[0] if pd.notna(r1['Count'].values[0]) else 0
            c2 = r2['Count'].values[0] if pd.notna(r2['Count'].values[0]) else 0
            xp.append(max(c1, 0.4)); yp.append(max(c2, 0.4))
    mk = 'o' if m in GPT else 's'
    ax_e.scatter(xp, yp, s=10, c=MC.get(m, C_GY), marker=mk,
                 edgecolors='white', lw=0.15, alpha=0.85, zorder=3, label=m)

bc_x = [BIOCHIRP[q] for q in QUESTIONS]
ax_e.scatter(bc_x, bc_x, s=22, c=C_BC, marker='D', edgecolors='white', lw=0.25, zorder=4, label='BioChirp')

ax_e.set_xscale('log'); ax_e.set_yscale('log')
ax_e.set_xlabel('Run 1 — records'); ax_e.set_ylabel('Run 2 — records')
ax_e.set_xlim(0.25, 8000); ax_e.set_ylim(0.25, 8000)
ax_e.set_aspect('equal')
ax_e.legend(frameon=False, fontsize=FS-2.5, loc='lower right', markerscale=0.7,
            handletextpad=0.15, labelspacing=0.25, ncol=2, columnspacing=0.4)
panel_label(ax_e, 'e')
ax_e.set_title('Reproducibility: Run 1 vs Run 2', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (f) Error types by provider
# ───────────────────────────────────────────────────
ax_f = fig.add_subplot(outer[2, 1])

err_map = {}
for e in mcp_df[mcp_df['Status']=='Fail']['Error'].str.strip().dropna().unique():
    if 'Upstream' in str(e) and 'Availability' in str(e): err_map[e] = 'Service\nUnavail.'
    elif 'Upstream' in str(e): err_map[e] = 'Retrieval\nFailure'
    elif 'MaxTurns' in str(e): err_map[e] = 'MaxTurns\nExceeded'
    elif 'BadRequest' in str(e): err_map[e] = 'Context\nOverflow'
    elif 'Timeout' in str(e): err_map[e] = 'MCP\nTimeout'
    else: err_map[e] = 'Other'

errs = mcp_df[mcp_df['Status']=='Fail'].copy()
errs['EType'] = errs['Error'].str.strip().map(err_map)
ecp = errs.groupby(['Provider','EType']).size().unstack(fill_value=0)

eorder = ['Retrieval\nFailure','MaxTurns\nExceeded','Context\nOverflow','MCP\nTimeout','Service\nUnavail.','Other']
eorder = [e for e in eorder if e in ecp.columns]

x_f = np.arange(len(eorder)); wf = 0.35
for i, prov in enumerate(['OpenAI','Claude']):
    if prov in ecp.index:
        vals = [ecp.loc[prov,e] if e in ecp.columns else 0 for e in eorder]
    else:
        vals = [0]*len(eorder)
    c = C_OA if prov=='OpenAI' else C_CL
    bars = ax_f.bar(x_f + (-wf/2 + i*wf), vals, wf*0.88, color=c, edgecolor='white', lw=0.2, label=prov, zorder=3)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax_f.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(v),
                      ha='center', fontsize=FS-2, fontweight='bold')

ax_f.set_xticks(x_f)
ax_f.set_xticklabels(eorder, fontsize=FS-1.5)
ax_f.set_ylabel('Failed runs')
ax_f.legend(frameon=False, loc='upper right', fontsize=FS-1)
ax_f.annotate('BioChirp: 0 errors\nacross all runs',
              xy=(0.97, 0.7), xycoords='axes fraction', fontsize=FS-1.5, color=C_BC,
              fontweight='bold', ha='right',
              bbox=dict(boxstyle='round,pad=0.25', fc='#e8f5e9', ec=C_BC, lw=0.4, alpha=0.9))
panel_label(ax_f, 'f')
ax_f.set_title('MCP failure modes by provider', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (g) Latency box + strip (OpenAI only)
# ───────────────────────────────────────────────────
# Use inner gridspec for bottom row: g (narrow) + h (wide heatmap)
gs_bottom = outer[3, :].subgridspec(1, 2, wspace=0.32, width_ratios=[0.38, 0.62])
ax_g = fig.add_subplot(gs_bottom[0])

lat_d, lat_l, lat_c = [], [], []
for m in GPT:
    sub = mcp_df[(mcp_df['Model']==m)&(mcp_df['Latency'].notna())]
    if len(sub) > 0:
        lat_d.append(sub['Latency'].values)
        lat_l.append(m); lat_c.append(MC[m])

bp = ax_g.boxplot(lat_d, patch_artist=True, widths=0.5,
    medianprops=dict(color='black', lw=0.6),
    whiskerprops=dict(lw=0.4), capprops=dict(lw=0.4),
    flierprops=dict(marker='o', ms=1.5, mfc=C_GY, mew=0.15), zorder=3)

for patch, c in zip(bp['boxes'], lat_c):
    patch.set_facecolor(c); patch.set_alpha(0.65); patch.set_edgecolor('white'); patch.set_linewidth(0.25)

np.random.seed(42)
for i, m in enumerate(lat_l):
    sub = mcp_df[(mcp_df['Model']==m)&(mcp_df['Latency'].notna())]
    succ = sub[sub['Status']=='Pass']['Latency'].values
    fail = sub[sub['Status']!='Pass']['Latency'].values
    js = np.random.uniform(-0.1, 0.1, len(succ))
    jf = np.random.uniform(-0.1, 0.1, len(fail))
    ax_g.scatter(np.full(len(succ), i+1)+js, succ, s=6, c=C_PASS, marker='o', edgecolors='white', lw=0.15, alpha=0.8, zorder=4)
    ax_g.scatter(np.full(len(fail), i+1)+jf, fail, s=6, c=C_FAIL, marker='x', lw=0.35, alpha=0.6, zorder=4)

ax_g.set_xticklabels(lat_l, rotation=35, ha='right')
ax_g.set_ylabel('Latency (seconds)')
lh = [Line2D([0],[0], marker='o', color='w', mfc=C_PASS, ms=3, label='Pass'),
      Line2D([0],[0], marker='x', color=C_FAIL, ms=3, mew=0.5, label='Fail', ls='')]
ax_g.legend(handles=lh, frameon=False, loc='upper left', fontsize=FS-1.5)
panel_label(ax_g, 'g')
ax_g.set_title('Query latency (OpenAI MCP)', loc='left', pad=3)

# ───────────────────────────────────────────────────
# (h) Heatmap — Best records per query–model pair
# ───────────────────────────────────────────────────
ax_h = fig.add_subplot(gs_bottom[1])

hm_models = ALL_MCP + ['BioChirp']
matrix = np.full((len(QUESTIONS), len(hm_models)), np.nan)
annot = np.empty((len(QUESTIONS), len(hm_models)), dtype=object)

for j, m in enumerate(hm_models):
    for i, q in enumerate(QUESTIONS):
        if m == 'BioChirp':
            matrix[i,j] = BIOCHIRP[q]; annot[i,j] = f'{BIOCHIRP[q]:,}'
        else:
            sub = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)]
            passes = sub[sub['Status']=='Pass']
            partials = sub[sub['Status']=='Partial']
            if len(passes) > 0:
                best = passes['Count'].max()
                matrix[i,j] = best; annot[i,j] = f'{int(best)}'
            elif len(partials) > 0:
                matrix[i,j] = 0.5; annot[i,j] = '~1'
            else:
                matrix[i,j] = 0; annot[i,j] = '✗'

# Log-normalize
log_m = np.log10(matrix + 1)
log_max = np.log10(max(BIOCHIRP.values()) + 1)

cmap = LinearSegmentedColormap.from_list('rg', [
    (0, '#fde0dc'), (0.001, '#fde0dc'), (0.015, '#fff3cd'),
    (0.08, '#d4edda'), (0.25, '#82c985'), (0.55, '#3a9e5c'), (1.0, '#145a32')])

im = ax_h.imshow(log_m / log_max, cmap=cmap, aspect='auto', vmin=0, vmax=1)

for i in range(len(QUESTIONS)):
    for j in range(len(hm_models)):
        txt = annot[i,j]
        brightness = log_m[i,j] / log_max
        color = 'white' if brightness > 0.45 else '#333333'
        if txt == '✗': color = C_FAIL
        w = 'bold' if hm_models[j] == 'BioChirp' else 'normal'
        ax_h.text(j, i, txt, ha='center', va='center', fontsize=FS-1.5, color=color, fontweight=w)

ax_h.set_xticks(np.arange(len(hm_models)))
ax_h.set_xticklabels(hm_models, rotation=40, ha='right', fontsize=FS-1)
ax_h.set_yticks(np.arange(len(QUESTIONS)))
ax_h.set_yticklabels([Q_SHORT[q] for q in QUESTIONS], fontsize=FS-0.5)

ax_h.axvline(4.5, color='white', lw=1.2)
ax_h.axvline(6.5, color='white', lw=1.2)

ax_h.text(2, -0.75, 'OpenAI', ha='center', fontsize=FS-0.5, color=C_OA, fontweight='bold')
ax_h.text(5.5, -0.75, 'Claude', ha='center', fontsize=FS-0.5, color=C_CL, fontweight='bold')
ax_h.text(7, -0.75, 'BioChirp', ha='center', fontsize=FS-0.5, color=C_BC, fontweight='bold')

panel_label(ax_h, 'h', x=-0.08)
ax_h.set_title('Best records retrieved (✗ = all runs failed)', loc='left', pad=12, fontsize=FS+0.5)

# ═══════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════
fig.savefig('BioChirp_vs_MCP_A4.png', dpi=300, facecolor='white')
fig.savefig('BioChirp_vs_MCP_A4.pdf', dpi=300, facecolor='white')
print("✓ Saved BioChirp_vs_MCP_A4.png and .pdf")

✓ Saved BioChirp_vs_MCP_A4.png and .pdf


In [3]:
"""
BioChirp vs MCP — Nature-Quality A4 Multi-Panel Figure (v3)
============================================================
8 panels on A4, white bg, Liberation Sans 7pt, soft Nature palette.
Includes Agentic vs ChatEndPoint comparison for OpenAI.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════
df = pd.read_excel('result_mcp.xlsx')

def classify(e):
    if pd.isna(e): return np.nan
    e = str(e).strip()
    if e == 'Pass': return 'Pass'
    elif 'Partial' in e: return 'Partial'
    return 'Fail'

df['Status'] = df['Error'].apply(classify)

Q_MAP = {
    'CML complete targets': 'CML\n(complete)',
    'CML top targets': 'CML\n(top)',
    'Top diseases treated with aspirin': 'Aspirin\n(top)',
    'All diseases treated with aspirin': 'Aspirin\n(all)',
    'Top diseases associated with TP53': 'TP53\n(top)',
    'All diseases associated with TP53': 'TP53\n(all)',
}
Q_SHORT = {k: v.replace('\n',' ') for k,v in Q_MAP.items()}
QUESTIONS = list(Q_MAP.keys())

def get_provider(m):
    if m in ['Haiku 3.5','Sonnet 4.5']: return 'Claude'
    elif m == 'biochirp': return 'BioChirp'
    return 'OpenAI'

df['Provider'] = df['Model'].apply(get_provider)
mcp_df = df[df['Model'] != 'biochirp'].copy()

BIOCHIRP = {
    'CML complete targets':4916, 'CML top targets':4916,
    'Top diseases treated with aspirin':363, 'All diseases treated with aspirin':363,
    'Top diseases associated with TP53':3258, 'All diseases associated with TP53':3258,
}
GPT = ['gpt-4.1-nano','gpt-4o-mini','gpt-4.1-mini','gpt-5-nano','gpt-5-mini']
CLAUDE = ['Haiku 3.5','Sonnet 4.5']
ALL_MCP = GPT + CLAUDE

def srate(sub):
    t = len(sub[sub['Status'].notna()])
    return (sub['Status']=='Pass').sum()/t*100 if t>0 else np.nan

# ═══════════════════════════════════════════════════
# STYLE — Nature soft palette, 7pt, white
# ═══════════════════════════════════════════════════
FS = 7

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Liberation Sans','Arial','Helvetica','DejaVu Sans'],
    'font.size': FS,
    'axes.titlesize': FS + 0.5,
    'axes.labelsize': FS,
    'xtick.labelsize': FS - 0.5,
    'ytick.labelsize': FS - 0.5,
    'legend.fontsize': FS - 1.5,
    'axes.linewidth': 0.35,
    'xtick.major.width': 0.35,
    'ytick.major.width': 0.35,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 0.6,
    'patch.linewidth': 0.25,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.03,
    'pdf.fonttype': 42,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'text.color': '#1a1a2e',
    'axes.labelcolor': '#1a1a2e',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.edgecolor': '#555555',
})

# Soft Nature palette
C_BC   = '#4caf50'   # soft green
C_CL   = '#5c9fd4'   # soft blue
C_OA   = '#e8935a'   # soft orange
C_FAIL = '#d47272'   # muted red
C_PART = '#c9b84a'   # muted olive
C_PASS = '#6abf7b'   # soft green
C_GY   = '#999999'
C_LG   = '#d5d5d5'
C_AGENT = '#c97a5a'  # warm terracotta for agentic
C_CHAT  = '#7aabd4'  # cool slate-blue for chatendpoint

# Soft model palette — warm oranges for GPT, cool blues for Claude
MC = {
    'gpt-4.1-nano':'#f4c49a', 'gpt-4o-mini':'#f0a86e', 'gpt-4.1-mini':'#de8a50',
    'gpt-5-nano':'#c46e3a', 'gpt-5-mini':'#9e5028',
    'Haiku 3.5':'#a3cbe5', 'Sonnet 4.5':'#5a9fd4', 'BioChirp':C_BC,
}

def plabel(ax, lbl, x=-0.13, y=1.08):
    ax.text(x, y, lbl, transform=ax.transAxes, fontsize=FS+2.5,
            fontweight='bold', va='top', ha='left', color='#1a1a2e')

# ═══════════════════════════════════════════════════
# A4 FIGURE: 210 × 297 mm
# ═══════════════════════════════════════════════════
A4W = 210/25.4
A4H = 297/25.4

fig = plt.figure(figsize=(A4W, A4H), facecolor='white')

# 4 rows × 2 cols; row 4 = heatmap full-width
outer = gridspec.GridSpec(4, 2, figure=fig,
    hspace=0.58, wspace=0.36,
    left=0.09, right=0.96, top=0.96, bottom=0.04,
    height_ratios=[1, 1, 1, 1.2])


# ──────────────────────────────────────────────
# (a) Success rate by model — horizontal bars
# ──────────────────────────────────────────────
ax_a = fig.add_subplot(outer[0, 0])
models_a = ALL_MCP + ['BioChirp']
rates_a, colors_a = [], []
for m in models_a:
    if m == 'BioChirp':
        rates_a.append(100.0); colors_a.append(C_BC)
    else:
        rates_a.append(srate(mcp_df[mcp_df['Model']==m]))
        colors_a.append(C_CL if m in CLAUDE else C_OA)

y_a = np.arange(len(models_a))
ax_a.barh(y_a, rates_a, height=0.6, color=colors_a, edgecolor='white', lw=0.2, zorder=3)
ax_a.set_yticks(y_a); ax_a.set_yticklabels(models_a)
ax_a.set_xlabel('Success rate (%)'); ax_a.set_xlim(0, 118)
ax_a.invert_yaxis()
ax_a.axvline(50, color=C_LG, lw=0.3, ls=':', zorder=1)

for i, v in enumerate(rates_a):
    if not np.isnan(v):
        w = 'bold' if models_a[i]=='BioChirp' else 'normal'
        c = '#1a1a2e' if v < 95 else '#2e7d32'
        ax_a.text(v+1.5, i, f'{v:.0f}%', va='center', fontsize=FS-1.5, fontweight=w, color=c)

# Bracket annotations
ax_a.axhline(4.5, color=C_LG, lw=0.35, ls=':', zorder=1)
ax_a.axhline(6.5, color=C_LG, lw=0.35, ls=':', zorder=1)
ax_a.text(108, 2, 'OpenAI', fontsize=FS-1.5, color=C_OA, ha='center', fontstyle='italic')
ax_a.text(108, 5.5, 'Claude', fontsize=FS-1.5, color=C_CL, ha='center', fontstyle='italic')
ax_a.text(108, 7, 'Ours', fontsize=FS-1.5, color=C_BC, ha='center', fontstyle='italic', fontweight='bold')

plabel(ax_a, 'a')
ax_a.set_title('Success rate across all queries', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# (b) Stacked outcome — MCP models + BioChirp
# ──────────────────────────────────────────────
ax_b = fig.add_subplot(outer[0, 1])
all_b = ALL_MCP + ['biochirp']
od = {'Pass':[], 'Partial':[], 'Fail':[]}
for m in all_b:
    sub = df[df['Model']==m]
    t = len(sub[sub['Status'].notna()])
    for s in ['Pass','Partial','Fail']:
        if m == 'biochirp':
            od[s].append(100.0 if s=='Pass' else 0.0)
        else:
            od[s].append((sub['Status']==s).sum()/t*100 if t>0 else 0)

x_b = np.arange(len(all_b))
bot = np.zeros(len(all_b))
cm = {'Pass':C_PASS, 'Partial':C_PART, 'Fail':C_FAIL}
for s in ['Pass','Partial','Fail']:
    ax_b.bar(x_b, od[s], 0.65, bottom=bot, color=cm[s], edgecolor='white', lw=0.15, label=s, zorder=3)
    bot += np.array(od[s])

labels_b = [m if m != 'biochirp' else 'BioChirp' for m in all_b]
ax_b.set_xticks(x_b); ax_b.set_xticklabels(labels_b, rotation=40, ha='right')
ax_b.set_ylabel('Proportion of runs (%)'); ax_b.set_ylim(0, 109)
ax_b.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5,1.02),
            handlelength=0.8, handletextpad=0.2, columnspacing=0.5)
ax_b.axvline(4.5, color=C_GY, lw=0.3, ls=':', zorder=2)
ax_b.axvline(6.5, color=C_GY, lw=0.3, ls=':', zorder=2)
ax_b.text(2, 105, 'OpenAI', ha='center', fontsize=FS-2, color=C_OA, fontstyle='italic')
ax_b.text(5.5, 105, 'Claude', ha='center', fontsize=FS-2, color=C_CL, fontstyle='italic')
ax_b.text(7, 105, 'BioChirp', ha='center', fontsize=FS-2, color=C_BC, fontstyle='italic', fontweight='bold')

plabel(ax_b, 'b')
ax_b.set_title('Run outcome distribution', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# (c) Provider success per query — with BioChirp
# ──────────────────────────────────────────────
ax_c = fig.add_subplot(outer[1, 0])
cr, oar = [], []
for q in QUESTIONS:
    cr.append(srate(mcp_df[(mcp_df['Provider']=='Claude')&(mcp_df['Question']==q)]))
    oar.append(srate(mcp_df[(mcp_df['Provider']=='OpenAI')&(mcp_df['Question']==q)]))

x_c = np.arange(len(QUESTIONS))
wc = 0.26
ax_c.bar(x_c - wc, oar, wc, color=C_OA, edgecolor='white', lw=0.15, label='OpenAI', zorder=3)
ax_c.bar(x_c, cr, wc, color=C_CL, edgecolor='white', lw=0.15, label='Claude', zorder=3)
ax_c.bar(x_c + wc, [100]*len(QUESTIONS), wc, color=C_BC, edgecolor='white', lw=0.15,
         label='BioChirp', zorder=3, alpha=0.75)

for vals, off in [(oar, -wc), (cr, 0)]:
    for i, v in enumerate(vals):
        if v > 0 and not np.isnan(v):
            ax_c.text(i+off+wc/2, v+2, f'{v:.0f}', ha='center', fontsize=FS-2.5, fontweight='bold', color='#333')

ax_c.set_xticks(x_c); ax_c.set_xticklabels([Q_MAP[q] for q in QUESTIONS])
ax_c.set_ylabel('Success rate (%)'); ax_c.set_ylim(0, 116)
ax_c.legend(frameon=False, ncol=3, fontsize=FS-1.5, loc='upper right',
            handlelength=0.7, handletextpad=0.2, columnspacing=0.4)

plabel(ax_c, 'c')
ax_c.set_title('Provider success rate per query', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# (d) Agentic vs ChatEndPoint — each model×endpoint as unique row
# ──────────────────────────────────────────────
ax_d = fig.add_subplot(outer[1, 1])

oai_df = mcp_df[(mcp_df['Provider']=='OpenAI') & (mcp_df['Type'].notna())].copy()

# Build list of (label, rate, endpoint_type) for each unique combo
combos = []
for m in GPT:
    for t in ['Agentic', 'ChatEndPoint']:
        sub = oai_df[(oai_df['Model']==m)&(oai_df['Type']==t)]
        r = srate(sub)
        tag = 'A' if t == 'Agentic' else 'C'
        combos.append((f'{m} ({tag})', r, t))

# Sort by success rate ascending (worst at top)
combos.sort(key=lambda x: (x[1] if not np.isnan(x[1]) else -1))

labels_d = [c[0] for c in combos]
rates_d = [c[1] for c in combos]
colors_d = [C_AGENT if c[2]=='Agentic' else C_CHAT for c in combos]

y_d = np.arange(len(combos))
ax_d.barh(y_d, rates_d, height=0.6, color=colors_d, edgecolor='white', lw=0.2, zorder=3)
ax_d.set_yticks(y_d); ax_d.set_yticklabels(labels_d, fontsize=FS-1.5)
ax_d.set_xlabel('Success rate (%)'); ax_d.set_xlim(0, 100)
ax_d.invert_yaxis()
ax_d.axvline(50, color=C_LG, lw=0.3, ls=':', zorder=1)

for i, v in enumerate(rates_d):
    if not np.isnan(v) and v > 0:
        ax_d.text(v+1.2, i, f'{v:.0f}%', va='center', fontsize=FS-2, fontweight='bold', color='#333')
    elif v == 0 or np.isnan(v):
        ax_d.text(1.5, i, '0%', va='center', fontsize=FS-2, color=C_FAIL)

# Legend
lh_d = [mpatches.Patch(fc=C_AGENT, label='Agentic (A)'),
        mpatches.Patch(fc=C_CHAT, label='ChatEndPoint (C)')]
ax_d.legend(handles=lh_d, frameon=False, fontsize=FS-1.5, loc='lower right',
            handlelength=0.8, handletextpad=0.25)

# Summary stats annotation
ax_d.annotate(
    'Agentic: 5 error types, median 36s\nChat: 3 error types, median 14s',
    xy=(0.97, 0.97), xycoords='axes fraction', fontsize=FS-2.5, color='#555',
    va='top', ha='right', linespacing=1.4,
    bbox=dict(boxstyle='round,pad=0.25', fc='#fdf6f0', ec='#ddd', lw=0.3, alpha=0.9))

plabel(ax_d, 'd')
ax_d.set_title('OpenAI: Agentic vs ChatEndPoint success', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# (e) Reproducibility scatter
# ──────────────────────────────────────────────
ax_e = fig.add_subplot(outer[2, 0])
ax_e.plot([0.3, 6000], [0.3, 6000], color=C_LG, lw=0.4, ls='--', zorder=1)

for m in ALL_MCP:
    xp, yp = [], []
    for q in QUESTIONS:
        r1 = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)&(mcp_df['Run']==1)]
        r2 = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)&(mcp_df['Run']==2)]
        if len(r1)>0 and len(r2)>0:
            c1 = r1['Count'].values[0] if pd.notna(r1['Count'].values[0]) else 0
            c2 = r2['Count'].values[0] if pd.notna(r2['Count'].values[0]) else 0
            xp.append(max(c1,0.4)); yp.append(max(c2,0.4))
    mk = 'o' if m in GPT else 's'
    ax_e.scatter(xp, yp, s=10, c=MC.get(m,C_GY), marker=mk,
                 edgecolors='white', lw=0.15, alpha=0.85, zorder=3, label=m)

bc_x = [BIOCHIRP[q] for q in QUESTIONS]
ax_e.scatter(bc_x, bc_x, s=24, c=C_BC, marker='D', edgecolors='white', lw=0.25, zorder=4, label='BioChirp')

ax_e.set_xscale('log'); ax_e.set_yscale('log')
ax_e.set_xlabel('Run 1 — records'); ax_e.set_ylabel('Run 2 — records')
ax_e.set_xlim(0.25, 8000); ax_e.set_ylim(0.25, 8000)
ax_e.set_aspect('equal')
ax_e.legend(frameon=False, fontsize=FS-2.5, loc='lower right', markerscale=0.7,
            handletextpad=0.1, labelspacing=0.2, ncol=2, columnspacing=0.3)

# Annotation for BioChirp
ax_e.annotate('BioChirp:\nperfect diagonal', xy=(4916, 4916), fontsize=FS-2,
              xytext=(100, 4500), color=C_BC, fontweight='bold',
              arrowprops=dict(arrowstyle='->', color=C_BC, lw=0.5),
              ha='left', va='top')

plabel(ax_e, 'e')
ax_e.set_title('Reproducibility: Run 1 vs Run 2', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# (f) Error types by provider
# ──────────────────────────────────────────────
ax_f = fig.add_subplot(outer[2, 1])

err_map = {}
for e in mcp_df[mcp_df['Status']=='Fail']['Error'].str.strip().dropna().unique():
    if 'Upstream' in str(e) and 'Availability' in str(e): err_map[e] = 'Upstream\nUnavailable'
    elif 'Upstream' in str(e): err_map[e] = 'Retrieval\nFailure'
    elif 'MaxTurns' in str(e): err_map[e] = 'MaxTurns\nExceeded'
    elif 'BadRequest' in str(e): err_map[e] = 'Context\nOverflow'
    elif 'Timeout' in str(e): err_map[e] = 'MCP\nTimeout'
    else: err_map[e] = 'Other'

errs = mcp_df[mcp_df['Status']=='Fail'].copy()
errs['EType'] = errs['Error'].str.strip().map(err_map)
ecp = errs.groupby(['Provider','EType']).size().unstack(fill_value=0)
eorder = ['Retrieval\nFailure','MaxTurns\nExceeded','Context\nOverflow','MCP\nTimeout','Upstream\nUnavailable','Other']
eorder = [e for e in eorder if e in ecp.columns]

x_f = np.arange(len(eorder)); wf = 0.33
for i, prov in enumerate(['OpenAI','Claude']):
    if prov in ecp.index:
        vals = [ecp.loc[prov,e] if e in ecp.columns else 0 for e in eorder]
    else:
        vals = [0]*len(eorder)
    c = C_OA if prov=='OpenAI' else C_CL
    bars = ax_f.bar(x_f + (-wf/2+i*wf), vals, wf*0.88, color=c, edgecolor='white', lw=0.15, label=prov, zorder=3)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax_f.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4, str(v),
                      ha='center', fontsize=FS-2, fontweight='bold', color='#333')

ax_f.set_xticks(x_f); ax_f.set_xticklabels(eorder, fontsize=FS-1.5)
ax_f.set_ylabel('Failed runs')
ax_f.legend(frameon=False, loc='upper right', fontsize=FS-1.5)
ax_f.annotate('BioChirp: 0 errors\nacross all runs', xy=(0.97, 0.68), xycoords='axes fraction',
              fontsize=FS-1.5, color='#2e7d32', fontweight='bold', ha='right',
              bbox=dict(boxstyle='round,pad=0.25', fc='#e8f5e9', ec=C_BC, lw=0.35, alpha=0.9))

plabel(ax_f, 'f')
ax_f.set_title('MCP failure modes by provider', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# Bottom row: (g) latency + (h) heatmap
# ──────────────────────────────────────────────
gs_bot = outer[3, :].subgridspec(1, 2, wspace=0.30, width_ratios=[0.42, 0.58])

# (g) Latency — per model×endpoint combo
ax_g = fig.add_subplot(gs_bot[0])

# Build combo latency data sorted same as panel d (by success rate)
combos_g = []
for m in GPT:
    for t in ['Agentic', 'ChatEndPoint']:
        sub = oai_df[(oai_df['Model']==m)&(oai_df['Type']==t)]
        r = srate(sub)
        lat = sub[sub['Latency'].notna()]['Latency'].values
        tag = 'A' if t == 'Agentic' else 'C'
        combos_g.append((f'{m} ({tag})', r, t, lat))

combos_g.sort(key=lambda x: (x[1] if not np.isnan(x[1]) else -1))

lat_data_g = [c[3] if len(c[3])>0 else [0] for c in combos_g]
lat_colors_g = [C_AGENT if c[2]=='Agentic' else C_CHAT for c in combos_g]
lat_labels_g = [c[0] for c in combos_g]

bp = ax_g.boxplot(lat_data_g, vert=False, patch_artist=True, widths=0.55,
    medianprops=dict(color='#333', lw=0.6),
    whiskerprops=dict(lw=0.35, color='#666'),
    capprops=dict(lw=0.35, color='#666'),
    flierprops=dict(marker='.', ms=1.5, mfc=C_GY, mew=0.1), zorder=3)

for patch, c in zip(bp['boxes'], lat_colors_g):
    patch.set_facecolor(c); patch.set_alpha(0.55); patch.set_edgecolor('white'); patch.set_linewidth(0.2)

ax_g.set_yticklabels(lat_labels_g, fontsize=FS-1.5)
ax_g.set_xlabel('Latency (seconds)')
ax_g.invert_yaxis()

lh = [mpatches.Patch(fc=C_AGENT, alpha=0.55, label='Agentic'),
      mpatches.Patch(fc=C_CHAT, alpha=0.55, label='ChatEndPoint')]
ax_g.legend(handles=lh, frameon=False, loc='lower right', fontsize=FS-2)

plabel(ax_g, 'g')
ax_g.set_title('Latency: Agentic vs Chat', loc='left', pad=3, fontweight='semibold')


# ──────────────────────────────────────────────
# (h) Heatmap
# ──────────────────────────────────────────────
ax_h = fig.add_subplot(gs_bot[1])

hm_models = ALL_MCP + ['BioChirp']
matrix = np.full((len(QUESTIONS), len(hm_models)), np.nan)
annot = np.empty((len(QUESTIONS), len(hm_models)), dtype=object)

for j, m in enumerate(hm_models):
    for i, q in enumerate(QUESTIONS):
        if m == 'BioChirp':
            matrix[i,j] = BIOCHIRP[q]; annot[i,j] = f'{BIOCHIRP[q]:,}'
        else:
            sub = mcp_df[(mcp_df['Model']==m)&(mcp_df['Question']==q)]
            passes = sub[sub['Status']=='Pass']
            partials = sub[sub['Status']=='Partial']
            if len(passes) > 0:
                best = passes['Count'].max(); matrix[i,j] = best; annot[i,j] = f'{int(best)}'
            elif len(partials) > 0:
                matrix[i,j] = 0.5; annot[i,j] = '~1'
            else:
                matrix[i,j] = 0; annot[i,j] = '✗'

log_m = np.log10(matrix + 1)
log_max = np.log10(max(BIOCHIRP.values()) + 1)

# Soft diverging colormap: rose → cream → sage → forest
cmap = LinearSegmentedColormap.from_list('soft', [
    (0.00, '#f2dede'),   # soft rose (fail)
    (0.001, '#f2dede'),
    (0.015, '#fef9e7'),  # warm cream
    (0.06, '#d5ecd4'),   # light sage
    (0.20, '#a3d9a5'),   # medium sage
    (0.50, '#5cb85c'),   # balanced green
    (1.00, '#1b5e20'),   # deep forest
])

im = ax_h.imshow(log_m / log_max, cmap=cmap, aspect='auto', vmin=0, vmax=1)

for i in range(len(QUESTIONS)):
    for j in range(len(hm_models)):
        txt = annot[i,j]
        brightness = log_m[i,j] / log_max
        color = 'white' if brightness > 0.42 else '#333333'
        if txt == '✗': color = '#c0392b'
        w = 'bold' if hm_models[j] == 'BioChirp' else 'normal'
        fs = FS - 1.5 if hm_models[j] != 'BioChirp' else FS - 0.5
        ax_h.text(j, i, txt, ha='center', va='center', fontsize=fs, color=color, fontweight=w)

ax_h.set_xticks(np.arange(len(hm_models)))
ax_h.set_xticklabels(hm_models, rotation=40, ha='right', fontsize=FS-1)
ax_h.set_yticks(np.arange(len(QUESTIONS)))
ax_h.set_yticklabels([Q_SHORT[q] for q in QUESTIONS], fontsize=FS-0.5)

# Provider dividers
ax_h.axvline(4.5, color='white', lw=1.5)
ax_h.axvline(6.5, color='white', lw=1.5)

ax_h.text(2, -0.7, 'OpenAI', ha='center', fontsize=FS-0.5, color=C_OA, fontweight='bold')
ax_h.text(5.5, -0.7, 'Claude', ha='center', fontsize=FS-0.5, color=C_CL, fontweight='bold')
ax_h.text(7, -0.7, 'BioChirp', ha='center', fontsize=FS-0.5, color=C_BC, fontweight='bold')

plabel(ax_h, 'h', x=-0.07)
ax_h.set_title('Best records retrieved  (✗ = all runs failed)', loc='left', pad=12, fontweight='semibold')

# ═══════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════
fig.savefig('BioChirp_vs_MCP_A4.png', dpi=300, facecolor='white')
fig.savefig('BioChirp_vs_MCP_A4.pdf', dpi=300, facecolor='white')
print("✓ Saved BioChirp_vs_MCP_A4.png and .pdf")

✓ Saved BioChirp_vs_MCP_A4.png and .pdf
